In [1]:
# IMPORTS & TARGET SCHEMA DEFINITION
import pandas as pd
import json
import time
import re
import difflib
import gradio as gr
from datetime import datetime

TARGET_SCHEMA = {
    "first_name": {
        "type": "string",
        "required": True,
        "aliases": ["fname", "first_name", "first", "given_name", "forename"]
    },
    "last_name": {
        "type": "string",
        "required": True,
        "aliases": ["lname", "last_name", "last", "surname", "family_name"]
    },
    "email": {
        "type": "string",
        "required": True,
        "aliases": ["email", "e-mail", "work_email", "mail", "mail_address", "email_id"]
    },
    "phone_number": {
        "type": "string",
        "required": False,
        "aliases": ["phone", "mobile", "cell", "contact_no", "telephone", "phone_number"]
    },
    "start_date": {
        "type": "date",
        "required": False,
        "aliases": ["start_date", "doj", "date_of_joining", "hire_date", "joining_date", "date"]
    },
    "status": {
        "type": "string",
        "required": True,
        "aliases": ["status", "emp_status", "active_status", "employment_status", "stat", "active"]
    }
}


In [2]:
# AI PROFILER & MULTI-SIGNAL CONFIDENCE ENGINE
def compute_column_confidence(col_name):
    col_clean = col_name.lower().replace("_", " ").replace("-", " ").strip()
    candidate_scores = []
    
    for field, spec in TARGET_SCHEMA.items():
        best_field_score = 0.0
        if any(alias == col_name.lower() or alias == col_clean for alias in spec["aliases"]):
            best_field_score = 1.0
        else:
            for alias in spec["aliases"]:
                alias_clean = alias.replace("_", " ")
                ratio = difflib.SequenceMatcher(None, col_clean, alias_clean).ratio()
                if alias_clean in col_clean or col_clean in alias_clean:
                    ratio = max(ratio, 0.88)
                if ratio > best_field_score:
                    best_field_score = ratio
        candidate_scores.append((field, round(best_field_score, 2)))
        
    candidate_scores.sort(key=lambda x: x[1], reverse=True)
    top_candidate, top_score = candidate_scores[0]
    second_candidate, second_score = candidate_scores[1] if len(candidate_scores) > 1 else ("NONE", 0.0)
    margin_of_victory = round(top_score - second_score, 2)
    
    passes_abs = top_score >= 0.85
    passes_mov = margin_of_victory >= 0.15
    is_autonomous = passes_abs and passes_mov
    
    return {
        "column": col_name,
        "top_candidate": top_candidate if is_autonomous else None,
        "top_score": top_score,
        "second_candidate": second_candidate,
        "second_score": second_score,
        "margin_of_victory": margin_of_victory,
        "passes_abs": passes_abs,
        "passes_mov": passes_mov,
        "is_autonomous": is_autonomous,
        "candidate_scores": candidate_scores
    }

def profile_and_map_columns(df):
    headers = df.columns.tolist()
    auto_mapped = {}
    escalations = []
    mapped_targets = set()
    
    for col in headers:
        eval_res = compute_column_confidence(col)
        if eval_res["is_autonomous"]:
            target = eval_res["top_candidate"]
            if target not in mapped_targets:
                auto_mapped[col] = target
                mapped_targets.add(target)
            else:
                escalations.append({
                    "context": f"AI Orchestration Conflict: Column '{col}' matches target '{target}' ({eval_res['top_score']*100:.0f}% confidence), but target already mapped.",
                    "record": json.dumps({
                        "problem_column": col,
                        "ai_suggestions": [target, "DROP"] + [k for k in TARGET_SCHEMA.keys() if k != target],
                        "selected_mapping": ""
                    }, indent=2)
                })
        else:
            reason = []
            if not eval_res["passes_abs"]:
                reason.append(f"Confidence ({eval_res['top_score']*100:.0f}%) < 85%")
            if not eval_res["passes_mov"]:
                reason.append(f"Margin of Victory ({eval_res['margin_of_victory']*100:.0f}%) < 15%")
            reason_str = " & ".join(reason)
            escalations.append({
                "context": f"AI Escalation Boundary Reached for '{col}': {reason_str}.",
                "record": json.dumps({
                    "problem_column": col,
                    "ai_suggestions": [eval_res['candidate_scores'][0][0], eval_res['second_candidate'], "DROP"],
                    "selected_mapping": ""
                }, indent=2)
            })
            
    return auto_mapped, escalations


In [3]:
# DETERMINISTIC ETL PIPELINE
def run_etl_pipeline(df, mapping):
    audit_trail = []
    
    mapping_clean = {k: v for k, v in mapping.items() if v and v != "DROP"}
    rev_mapping = {}
    for src, tgt in mapping_clean.items():
        if tgt not in rev_mapping:
            rev_mapping[tgt] = src
    final_rename = {v: k for k, v in rev_mapping.items()}
    df_mapped = df.rename(columns=final_rename)
    
    target_cols = [c for c in df_mapped.columns if c in TARGET_SCHEMA.keys()]
    df_clean = df_mapped[target_cols].copy()
    
    for field in TARGET_SCHEMA.keys():
        if field not in df_clean.columns:
            df_clean[field] = None
            
    audit_trail.append({"action": "Column Mapping", "details": f"Mapped {len(final_rename)} source columns to schema."})
    
    # Data Normalization
    if 'email' in df_clean.columns:
        def clean_email(val):
            if pd.isna(val):
                return None
            s = str(val).strip().lower()
            if s in ["", "nan", "null", "none", "n/a", "undefined"]:
                return None
            return s
        df_clean['email'] = df_clean['email'].apply(clean_email)
        audit_trail.append({"action": "Email Normalization", "details": "Lowercased and standardized null emails."})
        
    if 'start_date' in df_clean.columns:
        def clean_date(val):
            if pd.isna(val) or val is None:
                return None
            s = str(val).strip()
            if s in ["", "nan", "null", "none", "n/a"]:
                return None
            try:
                return pd.to_datetime(s).strftime('%Y-%m-%d')
            except Exception:
                return s
        df_clean['start_date'] = df_clean['start_date'].apply(clean_date)
        audit_trail.append({"action": "Date Normalization", "details": "Standardized dates to YYYY-MM-DD."})
        
    # Deduplication
    initial_len = len(df_clean)
    valid_email_mask = df_clean['email'].notna()
    df_with_email = df_clean[valid_email_mask].copy()
    df_without_email = df_clean[~valid_email_mask].copy()
    
    df_with_email_deduped = df_with_email.drop_duplicates(subset=['email'], keep='first')
    dupes_removed = len(df_with_email) - len(df_with_email_deduped)
    df_clean_deduped = pd.concat([df_with_email_deduped, df_without_email], ignore_index=True)
    audit_trail.append({"action": "Deduplication", "details": f"Removed {dupes_removed} duplicate email record(s)."})
    
    # Validation
    valid_records = []
    failed_records = []
    for idx, row in df_clean_deduped.iterrows():
        is_valid = True
        reasons = []
        for field, spec in TARGET_SCHEMA.items():
            val = row.get(field)
            is_null = pd.isna(val) or val is None or (isinstance(val, str) and val.strip() == "")
            if spec.get("required") and is_null:
                is_valid = False
                reasons.append(f"Required field '{field}' missing/null")
            if field == "start_date" and not is_null:
                if not re.match(r'^\d{4}-\d{2}-\d{2}$', str(val)):
                    is_valid = False
                    reasons.append(f"Invalid date format: {val}")
        if is_valid:
            valid_records.append(row.to_dict())
        else:
            failed_records.append({"row_index": idx + 1, "data": row.to_dict(), "reasons": reasons})
            
    audit_trail.append({"action": "Schema Validation", "details": f"{len(valid_records)} valid, {len(failed_records)} failed schema checks."})
    
    time.sleep(0.5)
    audit_trail.append({"action": "API Upload", "status": "201 Created", "records_inserted": len(valid_records)})
    
    metrics = {
        "processed": initial_len,
        "duplicates": dupes_removed,
        "failed": len(failed_records),
        "uploaded": len(valid_records)
    }
    
    return metrics, audit_trail, valid_records, failed_records


In [4]:
# GRADIO INTERFACE
theme = gr.themes.Default(primary_hue="indigo", neutral_hue="slate")

with gr.Blocks(theme=theme, title="Enterprise HCM Data Agent") as demo:
    gr.Markdown("# 🚀 Enterprise HCM Data Agent (MVP Architecture)")
    
    escalation_state = gr.State([])
    dataframe_state = gr.State(None)
    mapping_state = gr.State({})
    
    with gr.Tabs() as tabs:
        with gr.TabItem("1. Ingestion & Orchestration", id="tab1"):
            file_upload = gr.File(label="Upload HR Exports", file_types=[".xlsx", ".csv"])
            btn_trigger = gr.Button("Trigger Agentic Mapping", variant="primary")
            with gr.Row():
                agent_status = gr.Textbox(label="Agent Status", interactive=False)
                queue_status = gr.Textbox(label="Escalation Queue Metric", interactive=False)
            mapping_preview = gr.JSON(label="Autonomous Mappings (>85% Conf & >15% MoV)")
            btn_next_1 = gr.Button("Next ➡️ Go to Human Review")
            
        with gr.TabItem("2. Human-in-the-Loop Review", id="tab2"):
            agent_context = gr.Textbox(label="Agent Context (Boundary Escalation)", interactive=False)
            record_data = gr.Code(label="Escalation Decision Payload (JSON)", language="json", lines=12)
            with gr.Row():
                btn_load = gr.Button("Refresh / Load Escalation")
                btn_reject = gr.Button("Reject & Drop")
                btn_approve = gr.Button("Approve & Fix", variant="primary")
            btn_next_2 = gr.Button("Next ➡️ Run Target Sync & Audit")

        with gr.TabItem("3. Target Sync & Audit", id="tab3"):
            gr.Markdown("### ✅ Migration Dashboard")
            txt_summary = gr.Markdown("Waiting for pipeline completion...")
            json_audit = gr.JSON(label="Detailed Audit Trail")

    def trigger_agent(file, progress=gr.Progress()):
        if not file:
            return "No file uploaded.", "Empty", {}, [], None, {}
        df = pd.read_csv(file.name) if file.name.endswith('.csv') else pd.read_excel(file.name)
        auto_map, escalations = profile_and_map_columns(df)
        return f"Mapped {len(auto_map)} columns, {len(escalations)} escalated.", f"{len(escalations)} pending", auto_map, escalations, df, auto_map

    def get_current_escalation(queue):
        if not queue:
            return "✅ Queue is empty!", json.dumps({"status": "CLEAR"}, indent=2)
        return queue[0]["context"], queue[0]["record"]

    def approve_fix(json_data, queue, current_mapping):
        try:
            parsed = json.loads(json_data)
            col = parsed.get("problem_column")
            target = parsed.get("selected_mapping")
            if col and target:
                current_mapping[col] = target
            if queue:
                queue.pop(0)
            ctx, rec = get_current_escalation(queue)
            return ctx, rec, queue, current_mapping
        except Exception as e:
            return f"Invalid JSON: {e}", json_data, queue, current_mapping

    def reject_fix(queue, current_mapping):
        if queue:
            item = queue.pop(0)
            try:
                col = json.loads(item["record"]).get("problem_column")
                if col:
                    current_mapping[col] = "DROP"
            except Exception:
                pass
        ctx, rec = get_current_escalation(queue)
        return ctx, rec, queue, current_mapping

    def finalize_pipeline(df, queue, final_mapping):
        if df is None:
            return gr.update(selected="tab3"), "No data loaded.", []
        metrics, audit_trail, valid_recs, failed_recs = run_etl_pipeline(df, final_mapping)
        summary_md = f"""
### 📊 Migration Summary
| Metric | Count |
|---|---|
| Ingested Records | {metrics['processed']} |
| Duplicates Removed | {metrics['duplicates']} |
| Failed Validation | {metrics['failed']} |
| **Uploaded to API** | **{metrics['uploaded']}** |
"""
        return gr.update(selected="tab3"), summary_md, audit_trail

    btn_trigger.click(fn=trigger_agent, inputs=[file_upload], outputs=[agent_status, queue_status, mapping_preview, escalation_state, dataframe_state, mapping_state])
    btn_next_1.click(fn=lambda q: (gr.update(selected="tab2"), get_current_escalation(q)[0], get_current_escalation(q)[1]), inputs=[escalation_state], outputs=[tabs, agent_context, record_data])
    btn_load.click(fn=get_current_escalation, inputs=[escalation_state], outputs=[agent_context, record_data])
    btn_approve.click(fn=approve_fix, inputs=[record_data, escalation_state, mapping_state], outputs=[agent_context, record_data, escalation_state, mapping_state])
    btn_reject.click(fn=reject_fix, inputs=[escalation_state, mapping_state], outputs=[agent_context, record_data, escalation_state, mapping_state])
    btn_next_2.click(fn=finalize_pipeline, inputs=[dataframe_state, escalation_state, mapping_state], outputs=[tabs, txt_summary, json_audit])

demo.launch(height=700)

C:\Users\Vivek\AppData\Local\Temp\ipykernel_14568\4054354407.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Enterprise HCM Data Agent") as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


C:\Users\Vivek\AppData\Local\Temp\ipykernel_14568\2033012967.py:42: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s).strftime('%Y-%m-%d')
